In [36]:
import json
import joblib
import numpy as np
import os
import pandas as pd     
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, cohen_kappa_score


#import all algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
#xgboost is also classfication model but using xgboost library,
#  and it is combination of decision tree and gradient boosting
from xgboost import XGBClassifier

import pickle

In [37]:
#load the intent.json file
with open('intents.json') as f:
    data = json.load(f)

#store the patterns and tags in separate lists
#the patterns are the sentences that the user will input 
# and the tags are the labels that we want to predict
sentences = []
labels = []
for intent in data['intents']:
   
    for pattern in intent['patterns']:
        sentences.append(pattern)
        labels.append(intent['tag'])


print("total sentences: ", len(sentences))
print("total labels: ", len(labels))

total sentences:  257
total labels:  257


In [38]:
#train test split
#stratify is used to ensure that the distribution of labels in the train and test sets is the same as in the original dataset
#not using stratify will distribute the labels randomly and may lead to an imbalanced distribution of..... 
# labels in the train and test sets, which can affect the performance of the model
x_train, x_test, y_train, y_test = train_test_split(sentences, labels, test_size=0.2, random_state=42, stratify=labels)


print(f"Train text samples: {len(x_train)}")
print(f"Test text samples: {len(x_test)}")
print()


Train text samples: 205
Test text samples: 52



In [39]:
#tfidf vectorizer is used to convert the text(sentence) data into numerical data 
#One-Hot Encoding is used to encode categorical data (not text sentences) into binary (0/1) format.
vectorizer = TfidfVectorizer()
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)
print("trainin data shape: ", x_train_tfidf.shape)
print("testing data shape: ", x_test_tfidf.shape)


trainin data shape:  (205, 257)
testing data shape:  (52, 257)


In [40]:
#eveluate the model using different algorithms and compare their performance
def evelute_func(y_test,y_pred):
    acc=accuracy_score(y_test,y_pred)
    print("accurcy",acc)
    cm=confusion_matrix(y_test,y_pred)
    print("confusion metrx",cm)
    kp=cohen_kappa_score(y_test,y_pred)
    print("Kappa scor",kp)
    cr=classification_report(y_test,y_pred)
    print("classification report",cr)
    err=1-acc
    print("error rate",err)

In [41]:
# 1. Multinomial Naive Bayes
nb = MultinomialNB()
nb.fit(x_train_tfidf, y_train)
y_pred = nb.predict(x_test_tfidf)
evelute_func(y_test, y_pred)

accurcy 0.7307692307692307
confusion metrx [[6 0 0 0 0 0 0 0 0 0]
 [1 5 0 0 0 0 0 0 0 0]
 [1 0 2 0 0 0 0 0 1 0]
 [0 0 0 4 0 1 0 0 0 0]
 [2 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 4 0 0 1 0]
 [2 2 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 5 0 1]
 [1 0 0 0 0 0 0 0 3 1]
 [0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.6985507246376812
classification report                    precision    recall  f1-score   support

     crop_disease       0.46      1.00      0.63         6
       fertilizer       0.71      0.83      0.77         6
          goodbye       1.00      0.50      0.67         4
government_scheme       1.00      0.80      0.89         5
         greeting       1.00      0.50      0.67         4
       irrigation       0.80      0.80      0.80         5
             pest       1.00      0.20      0.33         5
             soil       1.00      0.83      0.91         6
      sowing_time       0.60      0.60      0.60         5
          weather       0.75      1.00      0.86         6

         accuracy      

In [42]:
#logestic regression
lr = LogisticRegression()
lr.fit(x_train_tfidf, y_train)
y_pred_lr = lr.predict(x_test_tfidf)
evelute_func(y_test,y_pred_lr)  

accurcy 0.7692307692307693
confusion metrx [[6 0 0 0 0 0 0 0 0 0]
 [1 4 0 0 0 0 0 0 1 0]
 [0 0 2 1 0 0 0 0 1 0]
 [0 0 0 4 0 1 0 0 0 0]
 [0 0 0 2 2 0 0 0 0 0]
 [0 0 0 0 0 4 0 0 1 0]
 [2 0 0 0 0 0 3 0 0 0]
 [0 0 0 0 0 0 0 6 0 0]
 [1 0 0 1 0 0 0 0 3 0]
 [0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.7423616845582164
classification report                    precision    recall  f1-score   support

     crop_disease       0.60      1.00      0.75         6
       fertilizer       1.00      0.67      0.80         6
          goodbye       1.00      0.50      0.67         4
government_scheme       0.50      0.80      0.62         5
         greeting       1.00      0.50      0.67         4
       irrigation       0.80      0.80      0.80         5
             pest       1.00      0.60      0.75         5
             soil       1.00      1.00      1.00         6
      sowing_time       0.50      0.60      0.55         5
          weather       1.00      1.00      1.00         6

         accuracy      

In [43]:
#svm
svm = SVC(kernel='linear')
svm.fit(x_train_tfidf, y_train)
y_pred_svm = svm.predict(x_test_tfidf)
evelute_func(y_test,y_pred_svm)

accurcy 0.8076923076923077
confusion metrx [[6 0 0 0 0 0 0 0 0 0]
 [1 4 0 0 0 0 0 0 1 0]
 [0 0 2 0 1 0 0 0 1 0]
 [0 0 0 4 0 1 0 0 0 0]
 [0 0 0 0 4 0 0 0 0 0]
 [0 0 0 0 0 4 0 0 1 0]
 [2 0 0 0 0 0 3 0 0 0]
 [0 0 0 0 0 0 0 6 0 0]
 [1 0 0 1 0 0 0 0 3 0]
 [0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.7855670103092783
classification report                    precision    recall  f1-score   support

     crop_disease       0.60      1.00      0.75         6
       fertilizer       1.00      0.67      0.80         6
          goodbye       1.00      0.50      0.67         4
government_scheme       0.80      0.80      0.80         5
         greeting       0.80      1.00      0.89         4
       irrigation       0.80      0.80      0.80         5
             pest       1.00      0.60      0.75         5
             soil       1.00      1.00      1.00         6
      sowing_time       0.50      0.60      0.55         5
          weather       1.00      1.00      1.00         6

         accuracy      

In [44]:
#random forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train_tfidf, y_train)
y_pred_rf = rf.predict(x_test_tfidf)
evelute_func(y_test,y_pred_rf)

accurcy 0.7884615384615384
confusion metrx [[5 0 0 0 0 0 1 0 0 0]
 [1 4 0 0 1 0 0 0 0 0]
 [0 0 2 0 2 0 0 0 0 0]
 [0 0 0 4 0 1 0 0 0 0]
 [0 0 0 0 4 0 0 0 0 0]
 [0 0 0 0 0 4 0 0 1 0]
 [0 1 0 0 1 0 3 0 0 0]
 [0 0 0 0 0 0 0 6 0 0]
 [1 0 0 1 0 0 0 0 3 0]
 [0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.7646090534979424
classification report                    precision    recall  f1-score   support

     crop_disease       0.71      0.83      0.77         6
       fertilizer       0.80      0.67      0.73         6
          goodbye       1.00      0.50      0.67         4
government_scheme       0.80      0.80      0.80         5
         greeting       0.50      1.00      0.67         4
       irrigation       0.80      0.80      0.80         5
             pest       0.75      0.60      0.67         5
             soil       1.00      1.00      1.00         6
      sowing_time       0.75      0.60      0.67         5
          weather       1.00      1.00      1.00         6

         accuracy      

In [46]:
#use k cross validation to find the best k for KNN for get the best performance of KNN
from sklearn.model_selection import cross_val_score

for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, x_train_tfidf, y_train, cv=5)
    print(k, scores.mean())

    #take the highest score and use that k to train the KNN model and evaluate it
    #here k=6 is the best k for KNN 0.4608....
#even k=8 is best we use k=7 because k=9 is even and it may lead to tie in the voting of KNN and we want to avoid that tie in voting
knn = KNeighborsClassifier(n_neighbors=9)
knn.fit(x_train_tfidf, y_train)
y_pred_knn = knn.predict(x_test_tfidf)
evelute_func(y_test,y_pred_knn)

1 0.697560975609756
2 0.6243902439024391
3 0.6634146341463415
4 0.6634146341463415
5 0.6878048780487805
6 0.6780487804878049
7 0.7170731707317074
8 0.7170731707317073
9 0.6829268292682927
10 0.6390243902439026
11 0.6634146341463415
12 0.6487804878048781
13 0.6487804878048781
14 0.6439024390243903
15 0.6341463414634146
16 0.6195121951219513
17 0.6195121951219512
18 0.6195121951219512
19 0.6048780487804878
20 0.6
accurcy 0.6538461538461539
confusion metrx [[6 0 0 0 0 0 0 0 0 0]
 [1 4 0 0 0 0 0 0 1 0]
 [1 0 2 1 0 0 0 0 0 0]
 [0 0 0 4 0 1 0 0 0 0]
 [2 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 4 0 1 0 0]
 [1 2 0 0 0 0 1 0 1 0]
 [0 1 0 0 0 0 1 4 0 0]
 [1 0 0 1 0 0 0 1 2 0]
 [0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.6125827814569537
classification report                    precision    recall  f1-score   support

     crop_disease       0.50      1.00      0.67         6
       fertilizer       0.57      0.67      0.62         6
          goodbye       1.00      0.50      0.67         4
government_scheme     

In [47]:
#decision tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train_tfidf, y_train)
y_pred_dt = dt.predict(x_test_tfidf)
evelute_func(y_test,y_pred_dt)

accurcy 0.7307692307692307
confusion metrx [[4 0 0 0 0 0 2 0 0 0]
 [1 3 0 1 0 0 0 0 0 1]
 [0 0 1 0 2 0 0 0 1 0]
 [0 0 0 4 0 1 0 0 0 0]
 [0 0 0 0 4 0 0 0 0 0]
 [0 0 0 0 0 4 0 0 1 0]
 [0 0 0 1 1 0 3 0 0 0]
 [0 1 0 0 0 0 0 5 0 0]
 [0 0 0 0 1 0 0 0 4 0]
 [0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.700657894736842
classification report                    precision    recall  f1-score   support

     crop_disease       0.80      0.67      0.73         6
       fertilizer       0.75      0.50      0.60         6
          goodbye       1.00      0.25      0.40         4
government_scheme       0.67      0.80      0.73         5
         greeting       0.50      1.00      0.67         4
       irrigation       0.80      0.80      0.80         5
             pest       0.60      0.60      0.60         5
             soil       1.00      0.83      0.91         6
      sowing_time       0.67      0.80      0.73         5
          weather       0.86      1.00      0.92         6

         accuracy       

In [48]:
#gradient boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(x_train_tfidf, y_train)
y_pred_gb = gb.predict(x_test_tfidf)
evelute_func(y_test,y_pred_gb)

accurcy 0.7884615384615384
confusion metrx [[5 0 0 0 0 0 1 0 0 0]
 [0 4 0 0 0 0 1 0 1 0]
 [0 0 2 0 0 0 2 0 0 0]
 [0 0 0 3 0 1 1 0 0 0]
 [0 0 0 0 2 0 2 0 0 0]
 [0 0 0 0 0 5 0 0 0 0]
 [0 0 0 0 0 0 5 0 0 0]
 [0 0 0 0 0 0 0 6 0 0]
 [1 0 0 0 0 0 1 0 3 0]
 [0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.7642209398186315
classification report                    precision    recall  f1-score   support

     crop_disease       0.83      0.83      0.83         6
       fertilizer       1.00      0.67      0.80         6
          goodbye       1.00      0.50      0.67         4
government_scheme       1.00      0.60      0.75         5
         greeting       1.00      0.50      0.67         4
       irrigation       0.83      1.00      0.91         5
             pest       0.38      1.00      0.56         5
             soil       1.00      1.00      1.00         6
      sowing_time       0.75      0.60      0.67         5
          weather       1.00      1.00      1.00         6

         accuracy      

In [49]:
#sgd
sgd = SGDClassifier(random_state=42)
sgd.fit(x_train_tfidf, y_train)
y_pred_sgd = sgd.predict(x_test_tfidf)    
evelute_func(y_test,y_pred_sgd)


accurcy 0.8269230769230769
confusion metrx [[6 0 0 0 0 0 0 0 0 0]
 [0 5 0 0 0 0 0 0 1 0]
 [0 0 3 0 0 0 0 0 1 0]
 [0 0 0 4 0 1 0 0 0 0]
 [0 0 2 0 2 0 0 0 0 0]
 [0 0 0 0 0 5 0 0 0 0]
 [1 0 0 0 0 0 3 0 1 0]
 [0 0 0 0 0 0 0 6 0 0]
 [1 0 0 0 0 0 0 1 3 0]
 [0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.8070103092783506
classification report                    precision    recall  f1-score   support

     crop_disease       0.75      1.00      0.86         6
       fertilizer       1.00      0.83      0.91         6
          goodbye       0.60      0.75      0.67         4
government_scheme       1.00      0.80      0.89         5
         greeting       1.00      0.50      0.67         4
       irrigation       0.83      1.00      0.91         5
             pest       1.00      0.60      0.75         5
             soil       0.86      1.00      0.92         6
      sowing_time       0.50      0.60      0.55         5
          weather       1.00      1.00      1.00         6

         accuracy      

In [ ]:
# XGBoost
from sklearn.preprocessing import LabelEncoder

# Encode labels once
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Train and predict
xgb = XGBClassifier(random_state=42)
xgb.fit(x_train_tfidf, y_train_encoded)
# Inverse transform predictions to original labels like 'greeting', 'goodbye', etc.
y_pred_xgb = le.inverse_transform(xgb.predict(x_test_tfidf))
evelute_func(y_test, y_pred_xgb)

accurcy 0.6538461538461539
confusion metrx [[4 0 0 0 0 0 2 0 0 0]
 [1 4 0 0 0 0 0 0 0 1]
 [0 0 2 0 2 0 0 0 0 0]
 [0 0 0 3 1 1 0 0 0 0]
 [0 0 0 0 4 0 0 0 0 0]
 [0 0 0 0 0 4 0 0 1 0]
 [0 1 0 0 2 0 2 0 0 0]
 [0 0 0 0 0 0 0 6 0 0]
 [0 0 0 1 2 0 1 0 0 1]
 [0 0 0 0 1 0 0 0 0 5]]
Kappa scor 0.6156057494866529
classification report                    precision    recall  f1-score   support

     crop_disease       0.80      0.67      0.73         6
       fertilizer       0.80      0.67      0.73         6
          goodbye       1.00      0.50      0.67         4
government_scheme       0.75      0.60      0.67         5
         greeting       0.33      1.00      0.50         4
       irrigation       0.80      0.80      0.80         5
             pest       0.40      0.40      0.40         5
             soil       1.00      1.00      1.00         6
      sowing_time       0.00      0.00      0.00         5
          weather       0.71      0.83      0.77         6

         accuracy      

In [53]:
#find the best model and save the best model and vectorizer using joblib
results = {
    "Naive Bayes"         : accuracy_score(y_test, y_pred),
    "Logistic Regression" : accuracy_score(y_test, y_pred_lr),
    "SVM"                 : accuracy_score(y_test, y_pred_svm),
    "Random Forest"       : accuracy_score(y_test, y_pred_rf),
    "KNN"                 : accuracy_score(y_test, y_pred_knn),
    "Decision Tree"       : accuracy_score(y_test, y_pred_dt),
    "Gradient Boosting"   : accuracy_score(y_test, y_pred_gb),
    "SGD"                 : accuracy_score(y_test, y_pred_sgd),
}

best_name = max(results, key=results.get)
print(f"Best model: {best_name} — {results[best_name]*100:.1f}%")



Best model: SGD — 82.7%


In [57]:
#save the best model
best_models = {
    "Naive Bayes": nb, "Logistic Regression": lr,
    "SVM": svm, "Random Forest": rf, "KNN": knn,
    "Decision Tree": dt, "Gradient Boosting": gb, "SGD": sgd
}
best_clf = best_models[best_name]
joblib.dump(best_clf, "model.pkl")


#save the vectorizer because we need to use the same vectorizer to 
# transform the user input before making predictions with the model
joblib.dump(vectorizer,"vectorizer.pkl")
print(f"Saved best model: {best_name} with vectorizer file")


Saved best model: SGD with vectorizer file
